In [22]:
import numpy as np
from collections import Counter
import pandas as pd
from sklearn import decomposition
from sklearn.metrics import confusion_matrix
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math 


In [24]:
class QuadTree: 
    def __init__(self, points, boundary, capacity=4, parent=None):  # capacity = max points before splitting
        self.parent = parent
        self.xmin, self.ymin, self.xmax, self.ymax = boundary # uses point to make boundery
        self.children = None # becomes list of 4 children if subdivided

        # Decide whether to make a leaf or subdivide
        if len(points) <= capacity or self.degenerate():
            self.points = points  # leaf node stores points
            self.size = len(points)
        else:
            self.points = None # only leafs store points
            self.subdivide(points, capacity)
            self.size = sum(child.size for child in self.children)

    def degenerate(self): # prevent self recursion
        return self.xmin == self.xmax or self.ymin == self.ymax

    def subdivide(self, points, capacity):
        mx = (self.xmin + self.xmax) / 2 # midpoint of x
        my = (self.ymin + self.ymax) / 2 # mispoint of y

        quads = [ 
            (self.xmin, self.ymin, mx, my),  # SW
            (mx, self.ymin, self.xmax, my),  # SE
            (self.xmin, my, mx, self.ymax),  # NW
            (mx, my, self.xmax, self.ymax)   # NE
        ]

        buckets = [[] for _ in range(4)] # hold points assigned to each quadrant 
        for x, y, label in points: # unpack points 
            for i, (xmin, ymin, xmax, ymax) in enumerate(quads):
                if xmin <= x <= xmax and ymin <= y <= ymax: # if in quadrant adds to list and does not check other quadrants
                    buckets[i].append((x, y, label))
                    break

        # Recursively build children
        self.children = [QuadTree(bucket, quads[i], capacity, parent=self) for i, bucket in enumerate(buckets)]

    def all_points(self):
        if self.points is not None:
            return self.points # if it is a leaf return all poinsts
        pts = []
        for child in self.children:
            pts.extend(child.all_points()) # gather all children points and return 
        return pts

    def quadrant_for_point(self, x, y):
        if self.children is None:
            return None
        for child in self.children:
            if child.xmin <= x <= child.xmax and child.ymin <= y <= child.ymax:
                return child
        return None
            
    def descend_for_k(self, x, y, k): # uses helper function to choose next child  
        node = self  
        while node.children is not None:
            child = node.quadrant_for_point(x,y)
            if child.size <k: # stops descending if child has fewer than k points
                return node
            node = child
        return node # returns deepest node that still has k points 
    
    def contains(self, x, y): # does this node's bounding box contain x,y
        return (self.xmin <= x <= self.xmax) and (self.ymin <= y <= self.ymax)
    
    def small_containing_quadtree(self, x, y): # return smallest quadtree with x, y 
        if not self.contains(x,y): # if does not contain point return empty 
            return None
        if self.children is None:
            return self
        for child in self.children:
            if child.contains(x,y):
                return child.small_containing_quadtree(x, y)
        return self 
    
    def within_distance(self, x, y, d):
        dx = 0
        if x < self.xmin:
            dx = self.xmin - x
        elif x > self.xmax:
            dx = x - self.xmax
        
        dy = 0 
        if y < self.ymin:
            dy = self.ymin - y
        elif y > self.ymax:
            dy = y - self.ymax
        return (dx*dx + dy*dy) <= d*d # if point is inside box, will return - if outside will be positive
    
    def leaves_within_distance(self, x, y, d, found=None):
        if found is None:
            found = []
        if not self.within_distance(x, y, d):
            return found
        if self.children is None:
            found.append(self)
            return found 
        for child in self.children:
            child.leaves_within_distance(x, y, d, found)
        return found 
    
def k_nearest_neighbors(tree, x0, y0, k, search_distance):
    leaves = tree.leaves_within_distance(x0, y0, d=search_distance) # leaves within search distance
    
    candidate_points = []
    for leaf in leaves:
        candidate_points.extend(leaf.all_points()) # gather all points from candidates
    if len(candidate_points) == 0:
        return [], [], None # if no points found 
    
    candidate_points = np.array(candidate_points)
    coords = candidate_points[:, :2].astype(float)

    dx = coords[:, 0] - x0 # distance using euclidean distances 
    dy = coords[:, 1] - y0
    distances = dx**2 + dy**2

    k_actual = min(k, len(candidate_points))
    min_i = np.argpartition(distances, k_actual-1)[:k_actual] # find k smallest distances 

    nearest_points = candidate_points[min_i]
    nearest_distances = distances[min_i]

    labels = [p[2] for p in nearest_points] # determine most common class label 
    predicted_class = Counter(labels).most_common(1)[0][0]

    return nearest_points, nearest_distances, predicted_class




In [25]:
# Used to test my tree - generated by ChatGPT
# Example points and boundary
points = [
    (10, 10, 'A'),
    (20, 15, 'B'),
    (42, 5, 'C'),
    (30, 25, 'D'),
    (50, 40, 'E')
]

# Define the boundary of your QuadTree: (xmin, ymin, xmax, ymax)
boundary = (0, 0, 60, 60)

# Create the QuadTree instance
tree = QuadTree(points, boundary, capacity=2)

# Now you can safely call leaves_within_distance
leaves = tree.leaves_within_distance(42, 17, d=20)

for leaf in leaves:
    print("Leaf size:", leaf.size)
    print("Points:", leaf.all_points())


Leaf size: 1
Points: [(20, 15, 'B')]
Leaf size: 1
Points: [(30, 25, 'D')]
Leaf size: 1
Points: [(42, 5, 'C')]
Leaf size: 0
Points: []
Leaf size: 1
Points: [(50, 40, 'E')]


In [26]:
nearest_pts, nearest_dists, predicted_class = k_nearest_neighbors(tree, 42, 6, 3, 20)

print("k nearest points:", nearest_pts)
print("Predicted class:", predicted_class)


k nearest points: [['42' '5' 'C']
 ['30' '25' 'D']
 ['20' '15' 'B']]
Predicted class: C


In [27]:
# call in rice
rice = pd.read_excel("Rice_Cammeo_Osmancik.xlsx")

In [28]:
rice.head()

,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,15231,525.578979,229.749878,85.093788,0.928882,15617,0.572896,Cammeo
1,14656,494.311005,206.020065,91.730972,0.895405,15072,0.615436,Cammeo
2,14634,501.122009,214.106781,87.768288,0.912118,14954,0.693259,Cammeo
3,13176,458.342987,193.337387,87.448395,0.891861,13368,0.640669,Cammeo
4,14688,507.166992,211.743378,89.312454,0.906691,15262,0.646024,Cammeo


In [29]:
# standardize 7 quantitative rice data
# used ChatGPT to standardize all except class one while retaining the class 

def standardize(series):
    return (series - series.mean()) / series.std()

rice_standard = rice.copy()
cols_to_standardize = [c for c in rice.columns if c !='Class']
rice_standard[cols_to_standardize] = rice[cols_to_standardize].apply(standardize)

rice_standard.head()

,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,1.479635,2.004091,2.348238,-0.212915,2.018073,1.499463,-1.152770,Cammeo
1,1.147720,1.125705,0.988261,0.945444,0.409964,1.192761,-0.602000,Cammeo
2,1.135020,1.317041,1.451718,0.253854,1.212797,1.126356,0.405558,Cammeo
3,0.293398,0.115285,0.261405,0.198025,0.239720,0.233826,-0.275315,Cammeo
4,1.166191,1.486858,1.316269,0.523351,0.952096,1.299685,-0.205986,Cammeo


In [30]:
pca_raw = decomposition.PCA() # performed on all components
pca_df = pd.DataFrame(
    pca_raw.fit_transform(rice_standard[cols_to_standardize])
)
pca_df = pca_df.rename(columns={0: 'PCA0', 1: 'PCA1'})
pca_df['Class'] = rice_standard['Class'].values

pca_df.head()

,PCA0,PCA1,2,3,4,5,6,Class
0,3.812128,2.165047,-0.117783,0.338934,0.255924,0.033872,0.001050,Cammeo
1,2.476833,-0.045290,-0.516482,-0.037646,-0.038593,0.037670,-0.017879,Cammeo
2,2.638209,0.621534,0.861792,0.078323,0.055707,0.017506,0.012085,Cammeo
3,0.547793,0.151383,-0.197109,-0.170814,0.035144,-0.050752,-0.004768,Cammeo
4,2.813662,0.482400,0.148023,0.155807,-0.067966,0.120758,-0.022692,Cammeo


In [ ]:
# plot PCA0 vs PCA1
# Used ChatGPT to fix errors thrown of using categorical classes in plotly

fig = px.scatter(
    x=pca_df['PCA0'],
    y=pca_df['PCA1'],
    color=rice['Class'],
    title="Rice Data (PC0 vs PC1)",
    labels={'x': 'PC0', 'y': 'PC1'},
    template='plotly_white'
)

fig.write_image('Original_PCA.png')
fig.show()


In [32]:
# asked chatGPT how to implement a test train split 
# used ChatGPT to only standardize for training, not test and understand to implement PCA after 

# Shuffle first
pca_df_shuffled = pca_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into features and labels
features = pca_df_shuffled.drop(columns='Class') # input values to use to predict
labels = pca_df_shuffled['Class'] # labels that are predicted

# 70/30 split - training on 70%, test on 30% data 
split_idx = int(0.7 * len(pca_df_shuffled))
X_train_raw = features.iloc[:split_idx].values
y_train = labels.iloc[:split_idx].values
X_test_raw = features.iloc[split_idx:].values
y_test = labels.iloc[split_idx:].values


# must train because KNN is a lazy alg - training is really just defining what points it can reference/giving comparison data to use 
# split into training to reference/compare and test which will check how well predicts for unseen points 

# Compute training mean and std
train_mean = X_train_raw.mean(axis=0)
train_std = X_train_raw.std(axis=0)

# Standardize
X_train = (X_train_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std  # IMPORTANT: use training mean/std

# Fit PCA only on training set
pca = decomposition.PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)

# Apply same PCA transformation to test set
X_test_pca = pca.transform(X_test)


In [33]:
# Combine X_train and y_train into list of (x, y, label)
train_points = [(x[0], x[1], label) for x, label in zip(X_train, y_train)]

# Determine the boundary of the QuadTree
xmin, ymin = X_train_pca.min(axis=0)
xmax, ymax = X_train_pca.max(axis=0)
boundary = (xmin, ymin, xmax, ymax)

# Build QuadTree
tree = QuadTree(train_points, boundary, capacity=2)


In [34]:
# looked at min and max of pca to choose good search distance 
print(pca_df.max())
print(pca_df.min())

PCA0     6.385878
PCA1     4.850452
2        3.034927
3        0.863499
4        0.706595
5        0.289397
6        0.149317
Class    Osmancik
dtype: object
PCA0    -5.571884
PCA1    -5.657457
2       -1.787305
3          -0.445
4       -0.418839
5       -0.156908
6       -0.115058
Class      Cammeo
dtype: object


In [35]:
# used ChatGPT to debug unknowns and add labels

y_pred = []
for x0, y0 in X_test_pca:
    _, _, pred = k_nearest_neighbors(tree, x0, y0, 1, 3)
    if pred is None:
        pred = 'Unknown'
    y_pred.append(pred)
y_pred = np.array(y_pred)

all_labels = np.unique(np.concatenate([y_test, y_pred]))

# Compute confusion matrix
cm_1 = confusion_matrix(y_test, y_pred, labels=all_labels)

# Make a nice DataFrame
cm_1_df = pd.DataFrame(cm_1, index=all_labels, columns=all_labels)
cm_1_df = cm_1_df.add_prefix('Predicted ').rename(index=lambda x: f'Was {x}')
print(cm_1_df)

              Predicted Cammeo  Predicted Osmancik
Was Cammeo                 306                 175
Was Osmancik               245                 417


In [36]:
# used ChatGPT to debug unknowns and add labels

y_pred = []
for x0, y0 in X_test_pca:
    _, _, pred = k_nearest_neighbors(tree, x0, y0, 5, 3)
    if pred is None:
        pred = 'Unknown'
    y_pred.append(pred)
y_pred = np.array(y_pred)

all_labels = np.unique(np.concatenate([y_test, y_pred]))

# Compute confusion matrix
cm_2 = confusion_matrix(y_test, y_pred, labels=all_labels)

# Make a nice DataFrame
cm_2_df = pd.DataFrame(cm_2, index=all_labels, columns=all_labels)
cm_2_df = cm_2_df.add_prefix('Predicted ').rename(index=lambda x: f'Was {x}')
print(cm_2_df)

              Predicted Cammeo  Predicted Osmancik
Was Cammeo                 319                 162
Was Osmancik               242                 420


In [37]:
# used ChatGPT to debug unknowns and add labels

y_pred = []
for x0, y0 in X_test_pca:
    _, _, pred = k_nearest_neighbors(tree, x0, y0, 1, 1)
    if pred is None:
        pred = 'Unknown'
    y_pred.append(pred)
y_pred = np.array(y_pred)

all_labels = np.unique(np.concatenate([y_test, y_pred]))

# Compute confusion matrix
cm_3 = confusion_matrix(y_test, y_pred, labels=all_labels)

# Make a nice DataFrame
cm_3_df = pd.DataFrame(cm_3, index=all_labels, columns=all_labels)
cm_3_df = cm_3_df.add_prefix('Predicted ').rename(index=lambda x: f'Was {x}')
print(cm_3_df)

              Predicted Cammeo  Predicted Osmancik  Predicted Unknown
Was Cammeo                 306                 174                  1
Was Osmancik               245                 411                  6
Was Unknown                  0                   0                  0


In [38]:
# used ChatGPT to debug unknowns and add labels

y_pred = []
for x0, y0 in X_test_pca:
    _, _, pred = k_nearest_neighbors(tree, x0, y0, 5, 1)
    if pred is None:
        pred = 'Unknown'
    y_pred.append(pred)
y_pred = np.array(y_pred)

all_labels = np.unique(np.concatenate([y_test, y_pred]))

# Compute confusion matrix
cm_3 = confusion_matrix(y_test, y_pred, labels=all_labels)

# Make a nice DataFrame
cm_3_df = pd.DataFrame(cm_3, index=all_labels, columns=all_labels)
cm_3_df = cm_3_df.add_prefix('Predicted ').rename(index=lambda x: f'Was {x}')
print(cm_3_df)

              Predicted Cammeo  Predicted Osmancik  Predicted Unknown
Was Cammeo                 319                 161                  1
Was Osmancik               243                 413                  6
Was Unknown                  0                   0                  0


In [39]:
# used ChatGPT to debug unknowns and add labels

y_pred = []
for x0, y0 in X_test_pca:
    _, _, pred = k_nearest_neighbors(tree, x0, y0, 1, 10)
    if pred is None:
        pred = 'Unknown'
    y_pred.append(pred)
y_pred = np.array(y_pred)

all_labels = np.unique(np.concatenate([y_test, y_pred]))

# Compute confusion matrix
cm_3 = confusion_matrix(y_test, y_pred, labels=all_labels)

# Make a nice DataFrame
cm_3_df = pd.DataFrame(cm_3, index=all_labels, columns=all_labels)
cm_3_df = cm_3_df.add_prefix('Predicted ').rename(index=lambda x: f'Was {x}')
print(cm_3_df)

              Predicted Cammeo  Predicted Osmancik
Was Cammeo                 306                 175
Was Osmancik               245                 417
